<h1 style="text-align: center; font-family: 'Roboto', sans-serif; font-weight:bold; font-size:40px; margin: 40px 0">05 - Modélisation</h1>

### Objectif
Ce notebook couvre la **modélisation prédictive** de l'attrition RH à partir du dataset consolidé `merged_data.csv`.
L'objectif est de construire un classifieur binaire (`has_left`) robuste en respectant les contraintes de fuite de données,
de déséquilibre de classe et d'interprétabilité.

### Plan de ce notebook
1. Importation des bibliothèques nécessaires
2. Chargement & inspection du dataset final
3. Sélection et préparation des features
4. Séparation Train / Test (stratifiée)
5. Preprocessing & Pipeline sklearn
6. Entraînement & Validation croisée stratifiée
7. Évaluation sur le test set
8. Interprétabilité du modèle
9. Réflexion Éthique

### Prérequis
- Avoir exécuté les notebooks [01](./01_eda_static_data.ipynb), [02](./02_eda_dynamic_data.ipynb), [03](./03_eda_merge_data.ipynb) et [04](./04_eda_analysis.ipynb).
- Le fichier `src/data/processed/merged_data.csv` doit exister.

---
## **1. Importation des bibliothèques nécessaires**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)

# TODO: Importer le ou les classifieur(s) que vous souhaitez comparer
# Exemples : RandomForestClassifier, LogisticRegression, GradientBoostingClassifier...
# from sklearn.??? import ???

# ── Chemins ──────────────────────────────────────────────────────────────────
DATA_FILE_PATH = '../data/processed/merged_data.csv'

# ── Configuration ML ─────────────────────────────────────────────────────────
RANDOM_STATE = 42
TEST_SIZE    = 0.2   # TODO: Ajuster si nécessaire
N_SPLITS     = 5     # Nombre de folds StratifiedKFold
TARGET_COL   = 'has_left'

# ── Styles visuels (cohérence avec les notebooks précédents) ─────────────────
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'DejaVu Sans',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
PALETTE     = {0: '#4A90D9', 1: '#E05C5C'}
SNS_PALETTE = ['#4A90D9', '#E05C5C']
LABELS      = {0: 'Resté (0)', 1: 'Parti (1)'}

print('Configuration chargée.')

---
## **2. Chargement & inspection du dataset final**

In [ ]:
df = pd.read_csv(DATA_FILE_PATH)
print(f"Dimensions : {df.shape}")
print(f"\nRépartition de la cible '{TARGET_COL}' :")
print(df[TARGET_COL].value_counts())
print(f"\nTaux d'attrition : {df[TARGET_COL].mean():.1%}")
df.head()

---
## **3. Sélection et préparation des features**

Avant de séparer les données, il faut décider quelles features conserver.
Le notebook 04 a mis en évidence une forte corrélation (`0.91`) entre `avg_hours_per_day` et `overtime_ratio`.
Conserver les deux features fortement corrélées peut fausser l'importance relative des variables dans certains modèles.

In [ ]:
# TODO: Identifier les features à exclure (multicolinéarité, features non informatives...)
# Indice : notebook 04, section 2.1 — matrice de corrélation
COLS_TO_DROP = [
    # TODO: Compléter cette liste
]

# Séparation features / cible
X = df.drop(columns=[TARGET_COL] + COLS_TO_DROP)
y = df[TARGET_COL]

# Identification automatique des types de colonnes
numeric_features     = X.select_dtypes(include='number').columns.tolist()
categorical_features = X.select_dtypes(exclude='number').columns.tolist()

print(f"Features numériques ({len(numeric_features)}) :")
print(f"  {numeric_features}")
print(f"\nFeatures catégorielles ({len(categorical_features)}) :")
print(f"  {categorical_features}")
print(f"\nX : {X.shape} | y : {y.shape}")

---
## **4. Séparation Train / Test (stratifiée)**

> ⚠️ **Règle absolue :** le split doit être réalisé **avant toute transformation**.
> Appliquer un `StandardScaler` ou un `SMOTE` sur l'ensemble complet constitue une **fuite de données** :
> le modèle exploiterait indirectement des informations du test set pendant l'entraînement.

In [ ]:
# Split AVANT toute transformation — stratify=y préserve les ~15% de positifs dans chaque fold
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

# Vérification de la stratification
print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"\nProportions '{TARGET_COL}' :")
print(f"  Total  : {y.mean():.1%}")
print(f"  Train  : {y_train.mean():.1%}")
print(f"  Test   : {y_test.mean():.1%}")

---
## **5. Preprocessing & Pipeline sklearn**

Un `Pipeline` chaîne les transformations et le classifieur en un seul objet.
Cela garantit que le `StandardScaler` est **fitté uniquement sur le train set** à chaque fold de cross-validation,
et appliqué (sans re-fit) sur le fold de validation et le test set final.

In [ ]:
# Preprocessing : StandardScaler sur numériques, OneHotEncoder sur catégorielles
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ],
    remainder='drop',
)

# TODO: Instancier votre classifieur
# Penser à class_weight='balanced' pour gérer le déséquilibre (~15% positifs)
# Commencer par un modèle simple avant d'augmenter la complexité
classifier = None  # TODO: Remplacer par votre classifieur

# Assemblage du Pipeline
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', classifier),
])

print("Pipeline créé :")
print(pipe)

---
## **6. Entraînement & Validation croisée stratifiée**

`StratifiedKFold` garantit que la proportion de la classe minoritaire (~15%) est conservée dans chaque fold.
`GridSearchCV` explore la grille d'hyperparamètres et sélectionne la combinaison optimisant la métrique choisie.

In [ ]:
# Validation croisée stratifiée
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

# TODO: Définir la grille d'hyperparamètres à explorer
# Les clés utilisent le format "étape__paramètre" du Pipeline
# Exemple pour RandomForest : 'classifier__n_estimators': [100, 200]
param_grid = {
    # 'classifier__???' : [...],
    # 'classifier__???' : [...],
}

# TODO: Choisir la métrique de scoring adaptée à l'objectif métier
# - 'recall'  : minimiser les faux négatifs (ne pas rater de départs)
# - 'f1'      : équilibre précision / recall
# - 'roc_auc' : performance globale sur tous les seuils
SCORING = '???'  # TODO: Choisir

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

search.fit(X_train, y_train)

print(f"\nMeilleur score CV ({SCORING}) : {search.best_score_:.4f}")
print(f"Meilleurs paramètres          : {search.best_params_}")

---
## **7. Évaluation sur le test set**

> ⚠️ Cette section ne doit être exécutée **qu'une seule fois**, à la toute fin.
> Ajuster le modèle après avoir consulté les métriques sur le test set constitue du **data snooping**.

In [ ]:
y_pred  = search.predict(X_test)
y_proba = search.predict_proba(X_test)[:, 1]

print("=" * 60)
print("RAPPORT DE CLASSIFICATION (test set)")
print("=" * 60)
print(classification_report(y_test, y_pred, target_names=['Resté (0)', 'Parti (1)']))
print(f"ROC-AUC : {roc_auc_score(y_test, y_proba):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Évaluation du modèle sur le test set", fontsize=14, fontweight='bold')

ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Resté (0)', 'Parti (1)'],
    cmap='Blues', ax=axes[0],
)
axes[0].set_title("Matrice de confusion")

RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], color=SNS_PALETTE[1])
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Aléatoire')
axes[1].set_title("Courbe ROC")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## **8. Interprétabilité du modèle**

Un modèle prédictif déployé en contexte RH doit être **explicable** : l'EU AI Act (Art. 13)
impose une transparence sur les facteurs influençant les décisions automatisées.
Les feature importances permettent d'identifier quelles variables pèsent le plus dans les prédictions.

In [ ]:
best_model = search.best_estimator_

# Récupération des noms de features après OneHotEncoding
feature_names = best_model.named_steps['preprocessor'].get_feature_names_out()

# TODO: Extraire les importances selon votre classifieur
# - Tree-based (RandomForest, GradientBoosting) : .feature_importances_
# - LogisticRegression : np.abs(.coef_[0])
importances = None  # TODO: Remplacer

if importances is not None:
    importance_df = (
        pd.DataFrame({'feature': feature_names, 'importance': importances})
        .sort_values('importance', ascending=False)
        .head(15)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(importance_df['feature'], importance_df['importance'],
            color=SNS_PALETTE[1], edgecolor='white')
    ax.set_title('Top 15 features les plus importantes', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance relative')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("TODO: Renseigner la variable 'importances' pour afficher le graphique.")

---
## **9. Réflexion Éthique**

### Équité algorithmique et biais de proxy

Les variables `department`, `job_role` et `education_field` peuvent être des **proxies de discrimination** :
certains rôles ou départements peuvent être historiquement associés à un genre ou une origine.
Un modèle entraîné sur des données de 2015 risque d'**amplifier des inégalités passées**.

Il est indispensable de vérifier que le recall du modèle est **uniforme entre les sous-groupes** :
si le modèle identifie mieux les départs dans un département que dans un autre, ses recommandations
de rétention seront inégalement distribuées.

### Transparence et droit à l'explication

Selon l'EU AI Act (Art. 13 & 14), les systèmes d'IA à haut risque (dont les outils RH) doivent :
- Fournir une **explication des décisions** aux personnes concernées.
- Permettre une **supervision humaine** avant toute décision de rétention ou de licenciement.

Le modèle prédictif doit donc rester un **outil d'aide à la décision**, jamais un outil de décision automatique.

### Question à se poser

> Avez-vous vérifié que le recall du modèle est comparable selon le `department` et le `job_role` ?
> Si un groupe présente un recall significativement plus faible, quelles actions correctrices envisagez-vous ?

---
<div style="display: flex; justify-content: space-between;">
  <a href="./04_eda_analysis.ipynb">Précédent</a>
  <a href=""></a>
</div>